# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Get top-level metadata as an object
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id and name
print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"  @id: {record_set['@id']} | name: {record_set.get('name', '(no name)')}")

# For each record set, list its fields and columns
for record_set in dataset.record_sets:
    print(f"\nRecord Set @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '(no name)')}")
    print("  Fields:")
    for field in record_set.get('field', []):
        if isinstance(field, dict):
            print(f"    - @id: {field.get('@id', field)}, name: {field.get('name', '(no name)')}")
        else:
            print(f"    - @id: {field}")
    print("  Columns:")
    for column in record_set.get('column', []):
        if isinstance(column, dict):
            print(f"    - @id: {column.get('@id', column)}, name: {column.get('name', '(no name)')}")
        else:
            print(f"    - @id: {column}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare: List the @id of all available record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record Set @id list:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id} with shape {df.shape}")
    else:
        print(f"No records found for record set {record_set_id}")

# Select a record set with data for further exploration
if dataframes:
    expl_record_set_id = list(dataframes.keys())[0]
    print(f"Using record set for EDA: {expl_record_set_id}")
    print("Columns:", dataframes[expl_record_set_id].columns.tolist())
    display(dataframes[expl_record_set_id].head())
else:
    print("No DataFrames loaded; the dataset may not include downloadable records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Assume we have a numeric field and a group field based on the previous overview
import numpy as np

if dataframes:
    df = dataframes[expl_record_set_id]
    # Try to automatically pick a numeric field
    numeric_field_candidates = [c for c in df.columns if df[c].dtype in [np.float64, np.float32, np.int64, np.int32]]
    if not numeric_field_candidates:
        # Try to find a numeric field by attempting conversion
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
                if df[c].notnull().sum() > 0:
                    numeric_field_candidates.append(c)
                    break
            except Exception:
                continue
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        # Filter and normalize as template
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by a likely group field
        group_field_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field detected in the DataFrame.")
else:
    print("No DataFrames loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dataframes and 'numeric_field' in locals():
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field or group field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.